In [ ]:
import pandas as pd
import numpy as np
import importlib
from statsmodels.stats.multitest import multipletests
import sys
import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import binomtest

# plot_utils.py lives in this folder (the parent of gene_level/), so make it importable.
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
import plot_utils as plot_utils
importlib.reload(plot_utils)
from plot_utils import plot_center_tails, plot_enhancer_stack_by_pct


In [ ]:
# --- Configuration Variables ---
# --- Input path (EDIT THIS) ---
# Master MPRA per-oligo annotation table (humanMPRA_annotations_v3.csv).
# On the lab server this was:
#   /home/labs/davidgo/Collaboration/humanMPRA/top_candidates/chondrocytes/humanMPRA_annotations_v3.csv
ANNOTATIONS_CSV = ""    # <- set to the path of humanMPRA_annotations_v3.csv

# Set your parameters here:

# Retain only genes linked to at least 'min_n' differentially active elements.
min_n = None                # Must be an integer >= 0
# Filter for elements that ATAC seq open in condrocytes.
condrocytes_filter = None  # Must be True or False
# Filter for elements that are top linked.
top_linked_filter = None   # Must be True or False

# --- Validation Logic ---
# This section ensures that parameters are filled correctly before proceeding.

# 1. Validate min_n (Type and Range)
if not isinstance(min_n, int) or min_n < 0:
    raise ValueError(f"Invalid Configuration: 'min_n' must be a non-negative integer. (Current value: {min_n})")

# 2. Validate condrocytes_filter (Type)
if not isinstance(condrocytes_filter, bool):
    raise TypeError(f"Invalid Configuration: 'condrocytes_filter' must be a Boolean (True/False). (Current type: {type(condrocytes_filter).__name__})")

# 3. Validate top_linked_filter (Type)
if not isinstance(top_linked_filter, bool):
    raise TypeError(f"Invalid Configuration: 'top_linked_filter' must be a Boolean (True/False). (Current type: {type(top_linked_filter).__name__})")

if not ANNOTATIONS_CSV:
    raise ValueError("Invalid Configuration: set ANNOTATIONS_CSV to the path of humanMPRA_annotations_v3.csv")

print("Configuration validated successfully. Proceeding with analysis...")


In [ ]:
fc_map = None  # Global variable to hold oligo ID to logFC mapping
    
def summarize_gene_oligos(df):
    """
    Group oligos by NCBI_Gene_ID and compute:
      - list of oligo IDs
      - list of logFC values
      - mean logFC (using safe_mean)
    """
    grouped = (
        df.groupby('NCBI_Gene_ID')['id']
          .agg(list)
          .reset_index()
    )
    grouped['logFC_list'] = grouped['id'].apply(oligos_to_logfc)
    grouped['mean_logFC'] = grouped['logFC_list'].apply(safe_mean)
    return grouped

def categorize_positive_pct(oligo_list):
    """
    Given a list of logFC values, return the percentage (0–100)
    of entries that are strictly > 0.
    Returns NaN if the input is None/NaN, not a sequence, or has no valid numeric values.
    """
    # Handle None / NaN (scalar float)
    if oligo_list is None or (isinstance(oligo_list, float) and pd.isna(oligo_list)):
        return np.nan

    # Expect a list/tuple/ndarray with non-zero length
    if not isinstance(oligo_list, (list, tuple, np.ndarray)) or len(oligo_list) == 0:
        return np.nan

    # Keep only numeric, non-NaN entries
    logfc_vals = [x for x in oligo_list
                  if isinstance(x, (int, float)) and not pd.isna(x)]

    if len(logfc_vals) == 0:
        return np.nan

    pct_positive = (sum(1 for x in logfc_vals if x > 0) / len(logfc_vals)) * 100
    return pct_positive


def bin_pct(pct):
    """
    Bin a percentage into discrete categories for plotting / summaries.
    NaN -> 'null'
    <= 0 -> '0%'
    (0, 25] -> '1–25%'
    (25, 50] -> '25–50%'
    (50, 75] -> '50–75%'
    (75, 99] -> '75–99%'
    > 99 -> '100%'
    """
    if pd.isna(pct):
        return 'null'
    if pct <= 0:
        return '0%'
    elif pct <= 25:
        return '1–25%'
    elif pct <= 50:
        return '25–50%'
    elif pct <= 75:
        return '50–75%'
    elif pct <= 99:
        return '75–99%'
    else:
        return '100%'
    

def oligos_to_logfc(oligo_list):
    """
    Map a list of oligo IDs to their logFC values using the global fc_map.
    Requires fc_map to be initialized later in the workflow.
    """
    if fc_map is None:
        raise ValueError("fc_map has not been initialized yet.")
    return [fc_map[o] for o in oligo_list if o in fc_map]

def safe_len(x):
    """Return len(x) if possible, otherwise 0."""
    try:
        return len(x)
    except:
        return 0


def safe_mean(x):
    """Return the mean of x, or NaN if x is empty.
    x is expected to be an iterable of numeric values.
    """
    return np.mean(x) if len(x) > 0 else np.nan


def safe_std(x):
    """
    Return the sample standard deviation (ddof=1) of x.
    - If x is not a list/tuple/ndarray or is empty -> NaN.
    - If x has length 1 -> 0.0 (by convention here).
    """
    if not isinstance(x, (list, tuple, np.ndarray)) or len(x) == 0:
        return np.nan

    if len(x) == 1:
        return 0.0

    return float(np.std(x, ddof=1))

def shuffle_nested_lists(nested_lists):
    """
    Receives a list of lists, returns a new list where each inner list
    is randomly shuffled (permuted independently).

    Parameters
    ----------
    nested_lists : list of lists
        A list containing sublists to be shuffled.

    Returns
    -------
    list of lists
        A new list where each inner list is shuffled.
    """
    result = []
    for lst in nested_lists:
        if lst is None or len(lst) == 0:
            result.append(lst)
            continue
        # convert to numpy array for permutation, then back to list
        shuffled = np.random.permutation(lst).tolist()
        result.append(shuffled)
    return result



def cumulative_nested_lists(nested_lists):
    """
    Receives a list of lists and returns a new list of lists where each
    inner list is replaced with its cumulative sum (computed using NumPy).

    Parameters
    ----------
    nested_lists : list of lists of numbers

    Returns
    -------
    list of lists
        Each inner list transformed into cumulative sum list.
    """
    result = []
    for lst in nested_lists:
        if lst is None or len(lst) == 0:
            result.append(lst)
            continue
        
        cum = np.cumsum(lst).tolist()
        result.append(cum)
    return result



def build_random_matrix(random_fc_lists):
    """
    Given a list of random trajectories (all assumed same length),
    build a 2D numpy array R with shape (n_random, L), containing the
    ABSOLUTE values of the trajectories.

    Returns:
      R (np.ndarray): shape (n_random, max_len)
    """
    max_len = max(len(traj) for traj in random_fc_lists)
    n_random = len(random_fc_lists)

    R = np.full((n_random, max_len), np.nan)

    for i, traj in enumerate(random_fc_lists):
        length = len(traj)
        R[i, :length] = np.abs(traj)

    return R


def empirical_p_value(value, rand_column, use_abs=True):
    rand = rand_column[~np.isnan(rand_column)]
    if len(rand) == 0:
        return np.nan

    if use_abs:
        value = abs(value)
        rand = np.abs(rand)

    count_more_extreme = np.sum(rand >= value)
    p = (count_more_extreme + 1) / (len(rand) + 1)

    return p






def compute_last_position_pvalues(
    df_gene_pool,
    random_paths,
    gene_id_col="NCBI_Gene_ID",
    traj_col="logFC_active",
    use_abs=True,
    debug=False
):
    """
    Compute empirical p-values for each gene based on the last point
    of its cumulative trajectory. A detailed debug mode is included.

    If debug=True → prints detailed diagnostic information.
    """

    # ---------------------------------------------
    # DEBUG: Input summary
    # ---------------------------------------------
    if debug:
        print("=============== DEBUG START ===============")
        print("Number of genes:", len(df_gene_pool))
        print("Number of random trajectories:", len(random_paths))
        print("Example random trajectory:", random_paths[0])
        print("Example gene trajectory:", df_gene_pool[traj_col].iloc[0])
        print("-------------------------------------------")

    # ---------------------------------------------
    # 1. CLEAN RANDOM TRAJECTORIES
    # ---------------------------------------------
    cleaned_random = []
    for traj in random_paths:
        if traj is None:
            continue
        filtered = [
            float(x) for x in traj
            if isinstance(x, (int, float)) and not pd.isna(x)
        ]
        if len(filtered) > 0:
            cleaned_random.append(filtered)

    if len(cleaned_random) == 0:
        raise ValueError("No valid random trajectories after cleaning.")

    # ---------------------------------------------
    # 2. BUILD RANDOM MATRIX R
    # ---------------------------------------------
    max_len = max(len(t) for t in cleaned_random)
    R = np.full((len(cleaned_random), max_len), np.nan)
    for i, traj in enumerate(cleaned_random):
        R[i, :len(traj)] = np.abs(traj)

    if debug:
        print("Random matrix shape:", R.shape)
        print("First row of R:", R[0])
        print("Max random trajectory length:", max_len)
        print("-------------------------------------------")

    # ---------------------------------------------
    # 3. PROCESS EACH GENE
    # ---------------------------------------------
    results = []

    for idx, row in df_gene_pool.iterrows():
        gene_id = row[gene_id_col]
        fc = row[traj_col]

        # Validate gene trajectory
        if not isinstance(fc, (list, tuple, np.ndarray)) or len(fc) == 0:
            if debug:
                print(f"[SKIP] Gene {gene_id} → invalid trajectory:", fc)
            continue

        L = len(fc)
        if L > max_len:
            if debug:
                print(f"[SKIP] Gene {gene_id} → gene length {L} exceeds random max {max_len}")
            continue

        last_val = fc[-1]
        if not isinstance(last_val, (int, float)) or pd.isna(last_val):
            if debug:
                print(f"[SKIP] Gene {gene_id} → invalid last value:", last_val)
            continue

        abs_last = abs(last_val) if use_abs else last_val

        # Extract random column
        rand_col = R[:, L - 1]
        valid_rand = rand_col[~np.isnan(rand_col)]

        if len(valid_rand) == 0:
            if debug:
                print(f"[SKIP] Gene {gene_id} → no random values at index {L-1}")
            continue

        # Compute p-value
        if use_abs:
            valid_rand_abs = np.abs(valid_rand)
            count_extreme = np.sum(valid_rand_abs >= abs_last)
        else:
            count_extreme = np.sum(valid_rand >= last_val)

        p = (count_extreme + 1) / (len(valid_rand) + 1)

        # ---------------------------------------------
        # DEBUG PRINT FOR FIRST 10 GENES
        # ---------------------------------------------
        if debug and idx < 10:
            print(f"\n--- DEBUG GENE {gene_id} ---")
            print("Gene length L:", L)
            print("Gene last value:", last_val)
            print("Absolute gene last value:", abs_last)
            print("Random values at index L-1:", valid_rand[:10])
            print("Min random:", valid_rand.min(), 
                  "Max random:", valid_rand.max(),
                  "Count random >= gene:", count_extreme)
            print("Computed p-value:", p)
            print("-------------------------------------------")

        results.append({
            gene_id_col: gene_id,
            "last_value": last_val,
            "abs_last_value": abs_last,
            "p_value": p,
            "L": L
        })

    # ---------------------------------------------
    # BUILD RESULT DF
    # ---------------------------------------------
    result_df = pd.DataFrame(results)
    result_df = result_df.sort_values("p_value").reset_index(drop=True)

    if debug:
        print("\n=============== DEBUG SUMMARY ===============")
        print("Total processed genes:", len(result_df))
        print("Genes with p < 0.05:", (result_df["p_value"] < 0.05).sum())
        print("=============================================")

    return result_df


def plot_genes(genes_fc_lists):
    """
    Plot two sets of FC trajectories on the same figure:

    1) Random FC trajectories (random_fc_lists):
       - Each element is a list of logFC values.
       - Plotted as light-blue dashed lines with high transparency.

    2) Gene FC trajectories (genes_fc_lists):
       - Each element is a list of logFC values for a single gene.
       - Plotted as purple lines with very high transparency.
       - IMPORTANT: No shuffling is applied; the order of each list
         is preserved exactly as given.

    Parameters
    ----------
    random_fc_lists : list of lists of floats
        A collection of random FC trajectories.

    genes_fc_lists : list of lists of floats
        A collection of gene FC trajectories.

    Notes
    -----
    - All non-numeric and NaN values are filtered out.
    - The X-axis represents the index within each FC list 
      (1, 2, 3, ..., N).
    - No legend is displayed.

    """
    
    fig, ax = plt.subplots(figsize=(12, 6))

    # -------------------------------------------------------
    # 1 Plot gene FC trajectories
    # -------------------------------------------------------
    gene_count = 0
    for i, path in enumerate(genes_fc_lists):
        if path is None or len(path) == 0:
            continue
        # Keep only valid numeric values
        y = [x for x in path if isinstance(x, (int, float)) and not np.isnan(x)]
        if len(y) == 0:
            continue

        x = np.arange(1, len(y) + 1)
        
        # Plot the shuffled FC trajectory
        x = np.arange(1, len(y) + 1)
        ax.plot(
            x,
            y,
            color='purple',
            alpha=0.3,
            linewidth=0.5
        )

        gene_count += 1

    # -------------------------------------------------------
    # Styling
    # -------------------------------------------------------
    ax.set_xlabel("Index in trajectory")
    ax.set_ylabel("logFC")
    ax.set_title(" Gene trajectories")

    # Remove legend if it exists
    legend = ax.get_legend()
    if legend is not None:
        legend.remove()

    plt.tight_layout()
    plt.show()


def split_col_to_list(s: pd.Series) -> pd.Series:
    """
    Convert a Series into a Series of lists.
    Handles:
        - NaN → []
        - numeric values → ["value"]
        - strings with delimiters: "A;B,C" → ["A","B","C"]
        - empty / whitespace → []
    """
    return (
        s.fillna("")                # remove NaN
         .astype(str)               # ensure all values are strings
         .str.strip()
         .replace("nan", "", regex=False)   # handle "nan" string
         .str.replace(";", ",", regex=False)  # unify delimiters
         .apply(lambda x: [g.strip() for g in x.split(",") if g.strip()])
    )


def split_cols_to_lists(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """
    Apply split_col_to_list() to multiple columns of a DataFrame.
    Returns a DataFrame where each listed column is a list of strings.
    """
    return df.assign(**{
        col: lambda d, c=col: split_col_to_list(d[c])
        for col in cols
    })



In [ ]:
def run_sign_test(data):
    """
    Performs a Sign Test to check if signs are uniformly distributed.
    H0: P(+) = 0.5 (Uniform distribution)
    H1: P(+) != 0.5 (Bias towards positive or negative)
    """
    
    # 1. Filter out zeros (ties are excluded in sign test)
    clean_data = [x for x in data if x != 0]
    n_total = len(clean_data)
    
    if n_total == 0:
        print("Error: All values are 0. Cannot perform test.")
        return

    # 2. Count positives and negatives
    n_positive = sum(x > 0 for x in clean_data)
    n_negative = sum(x < 0 for x in clean_data)
    
    # 3. Perform the Two-Sided Binomial Test
    # k = number of successes (positives), n = total trials, p = 0.5 (fair coin)
    result = binomtest(k=n_positive, n=n_total, p=0.5, alternative='two-sided')
    
    # 4. Output results
    print("--- Sign Test Results ---")
    print(f"Total N (excluding zeros): {n_total}")
    print(f"Positives (+): {n_positive}")
    print(f"Negatives (-): {n_negative}")
    print(f"P-value: {result.pvalue:}")
    
    # 5. Interpretation
    alpha = 0.05
    if result.pvalue < alpha:
        print(">> Conclusion: The distribution is NOT uniform (Statistically Significant Bias).")
    else:
        print(">> Conclusion: The distribution appears uniform (No significant bias found).")

In [ ]:
def plot_enhancer_stack_by_pct_grouped_visual(
    vals,
    pct_bins,
    xlabel="Number of diff-active enhancers per gene",
    title="Distribution of diff-active enhancers per gene",
):
    """
    Plot a stacked bar chart ordered by COLOR intensity rather than numerical value.
    Order: Null (Bottom) -> Middle -> Extreme (Top).
    """

    # 1. Define Colors
    color_null = '#E8F5E9'   # Very Light Green (Null)
    color_mid  = '#66BB6A'   # Medium Green (Middle)
    color_ext  = '#1B5E20'   # Dark Green (Extreme)

    # 2. Define Plotting Groups (VISUAL ORDER)
    # Instead of ordering 0% -> 100%, we order by visual grouping.
    # We merge the low extremes (0-25%) and high extremes (75-100%) into one block.
    plotting_groups = [
        # Group 1: Bottom (Lightest)
        {
            'label': 'Null', 
            'bins': ['null'], 
            'color': color_null
        },
        # Group 2: Middle (Medium intensity)
        {
            'label': 'Middle', 
            'bins': ['25–50%', '50–75%'], 
            'color': color_mid
        },
        # Group 3: Top (Darkest - combines both low and high extremes)
        {
            'label': 'Extreme', 
            'bins': ['0%', '1–25%', '75–99%', '100%'], 
            'color': color_ext
        },
    ]

    # Flatten list for data extraction
    all_possible_bins = [b for group in plotting_groups for b in group['bins']]

    # --- Data Preparation ---
    mask_valid = vals.notna() & pct_bins.notna()
    vals = vals[mask_valid]
    pct_bins = pct_bins[mask_valid]

    enhancer_counts = sorted(
        set(list(range(0, int(vals.dropna().max()) + 1)) + list(vals.dropna().unique()))
    )

    # Calculate raw counts
    data_raw = {pct: [] for pct in all_possible_bins}
    for enh_count in enhancer_counts:
        mask_count = vals == enh_count
        for pct in all_possible_bins:
            mask_pct = pct_bins == pct
            count = (mask_count & mask_pct).sum()
            data_raw[pct].append(count)

    # --- Plotting ---
    fig, ax = plt.subplots(figsize=(18, 6))
    x_pos = np.arange(len(enhancer_counts))
    bottom = np.zeros(len(enhancer_counts), dtype=int)

    # Loop through groups to stack them
    for group in plotting_groups:
        group_heights = np.zeros(len(enhancer_counts), dtype=int)
        
        # Sum all bins belonging to this visual group
        for bin_name in group['bins']:
            # Safety check if bin exists in data
            if bin_name in data_raw:
                group_heights += np.array(data_raw[bin_name])
        
        # Plot the consolidated block
        ax.bar(
            x_pos,
            group_heights,
            color=group['color'],
            edgecolor='black',
            linewidth=0.8,
            bottom=bottom,
        )
        bottom += group_heights

    # --- Annotations ---
    for i, total in enumerate(bottom):
        if total == 0: continue
        ax.text(x_pos[i], total + 0.5, str(int(total)), ha='center', va='bottom', fontsize=9)

    # Statistics
    count_non_na = int(vals.dropna().shape[0])
    mean_enh = vals.mean()
    median_enh = vals.median()
    std_enh = vals.std()

    summary_text = (
        f"Genes with valid enhancer count: {count_non_na}\n\n"
        f"Mean enhancers per gene: {mean_enh:.2f}\n\n"
        f"Median enhancers per gene: {median_enh}\n\n"
        f"Std Dev enhancers per gene: {std_enh:.2f}"
    )
    ax.text(1.02, 0.95, summary_text, transform=ax.transAxes, fontsize=12, va='top', fontweight='bold')

    ax.set_xlabel(xlabel)
    ax.set_ylabel("Number of genes")
    ax.set_title(title)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(enhancer_counts)

    # --- Custom Legend ---
    legend_patches = [
        mpatches.Patch(facecolor=color_null, edgecolor='black', label='Null (NaN)'),
        mpatches.Patch(facecolor=color_mid,  edgecolor='black', label='Middle (25%–75%)'),
        mpatches.Patch(facecolor=color_ext,  edgecolor='black', label='Extreme (<25% or >75%)'),
    ]
    
    ax.legend(handles=legend_patches, title="Status / % Positive", loc='upper right')

    plt.tight_layout()
    return fig, ax

In [ ]:
def split_col_to_list(s: pd.Series) -> pd.Series:
    """
    Convert a Series into a Series of lists.
    Handles:
        - NaN → []
        - numeric values → ["value"]
        - strings with delimiters: "A;B,C" → ["A","B","C"]
        - empty / whitespace → []
    """
    return (
        s.fillna("")                # remove NaN
         .astype(str)               # ensure all values are strings
         .str.strip()
         .replace("nan", "", regex=False)   # handle "nan" string
         .str.replace(";", ",", regex=False)  # unify delimiters
         .apply(lambda x: [g.strip() for g in x.split(",") if g.strip()])
    )


def split_cols_to_lists(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """
    Apply split_col_to_list() to multiple columns of a DataFrame.
    Returns a DataFrame where each listed column is a list of strings.
    """
    return df.assign(**{
        col: lambda d, c=col: split_col_to_list(d[c])
        for col in cols
    })



In [ ]:
def expand_rows_by_gene(df: pd.DataFrame, list_cols: list[str]) -> pd.DataFrame:
    """
    Expand rows where some columns contain lists (same length per row),
    so that each element in those lists gets its own row.

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame, where some columns hold list-like values.
    list_cols : list of str
        Column names that contain lists and are aligned by index
        (e.g. gene IDs, symbols, distances, flags per gene).

    Returns
    -------
    pd.DataFrame
        A new DataFrame where each gene (list element) gets its own row,
        with scalar columns duplicated and list columns split by index.
    """
    df = df.copy()

    # Create a single column that zips all list-columns together per row.
    # For each row, this becomes:
    #   [(val_col1_0, val_col2_0, ...),
    #    (val_col1_1, val_col2_1, ...),
    #    ...]
    df["__zipped__"] = df[list_cols].apply(
        lambda row: list(zip(*row)),  # each element is a tuple across columns
        axis=1
    )

    # Explode: each tuple in "__zipped__" becomes its own row
    df_expanded = df.explode("__zipped__", ignore_index=True)

    # Unpack the tuple back into the original list-columns
    for i, col in enumerate(list_cols):
        df_expanded[col] = df_expanded["__zipped__"].apply(lambda t: t[i])

    # Drop helper column
    df_expanded = df_expanded.drop(columns="__zipped__")

    return df_expanded


In [ ]:
# --- Initial Data Loading and Quality Control ---

# 1. Load the core annotation dataset (path set in the config cell above)
df_oligo = pd.read_csv(ANNOTATIONS_CSV)

# 2. Apply a sequencing depth filter:
#    We require at least 51 raw counts for both alleles (ancestral & derived).
#    This reduces the impact of sampling noise on the resulting ratios/FC.
df_oligo = df_oligo[(df_oligo["DNA_counts_raw_ancestral"]>50) & (df_oligo["DNA_counts_raw_derived"]>50)]

# 3. Display the filtered dataframe
df_oligo


In [ ]:
# --- Conditional Cell-Type Filtering ---

# Check if the chondrocyte-specific filter is enabled in the configuration
if condrocytes_filter:
    # Filter the oligos based on fetal chondrocyte ATAC-seq peak intensity.
    # We retain elements with values strictly between -100 and 100.
    df_oligo = df_oligo[
        (df_oligo['ATACseq_peaks_fetal_chondrocytes'] < 100) & 
        (df_oligo['ATACseq_peaks_fetal_chondrocytes'] > (-100))
    ]
    
    # Display the filtered dataframe for verification
    df_oligo

In [ ]:
# --- Column Selection and Gene ID Cleaning ---

df_oligo2 = (
    # 1. Column Slicing: Select the identifier (index 0) and the relevant annotation range (indices 7-23)
    df_oligo.iloc[:, [0] + list(range(7, 24))] 
    
    # 2. Deduplication: Remove redundant entries based on the primary oligo identifier
    .drop_duplicates(subset=df_oligo.columns[0]) 
    
    # 3. Gene ID Normalization:
    #    Convert messy NCBI_Gene_ID strings into clean, list-based formats.
    .assign(
        NCBI_Gene_ID=lambda d: (
            d["NCBI_Gene_ID"]
                .fillna("")                                 # Replace NaNs with empty strings
                .str.replace(";", ",", regex=False)         # Unify delimiters 
                # Split strings by comma and strip whitespace to create an array of IDs
                .apply(lambda x: [g.strip() for g in x.split(",") if g.strip()])
        )
    )
)

# Display the structured dataframe
df_oligo2

In [ ]:
# --- Filtered dataset containing only gene-linked oligos ---

# List of columns containing gene-specific data that need to be parsed into lists
cols_to_list = [
    "NCBI_Gene_ID",
    "Gene_symbol",
    "Distance_to_gene(TSS)",
    "closest_gene",
    "within_promoter",
    "eRNA",
    "eQTL",
    "HiC",
    "num_sources_linking_gene",
]

df_has_genes = (
    df_oligo
      # Select the identifier and relevant annotation range
      .iloc[:, [0] + list(range(7, 32))]   
      
      # CRITICAL FILTER: Keep only oligos linked to at least one gene.
      # This removes elements that cannot be mapped to a genomic target.
      .dropna(subset=["NCBI_Gene_ID"])     
      
      # Ensure unique entries before expansion
      .drop_duplicates(subset=df_oligo.columns[0])  
      
      # Convert string-delimited metadata into lists for multi-gene handling
      .pipe(split_cols_to_lists, cols=cols_to_list) 
)

# Display the filtered dataset containing only gene-linked oligos
df_has_genes

In [ ]:
# --- Regulatory Activity Filtering ---

# 1. Filter for elements with a defined 'differential_activity' status.
#    This removes elements where activity could not be determined (NaN).
df_active_and_diff = df_has_genes[
    (df_has_genes["differential_activity"].notna())
].copy()

# 2. Display the resulting subset focused on regulatory activity
df_active_and_diff

In [ ]:
# --- Dataset Expansion (Explode) ---

# 1. Expand the filtered 'active and diff' dataframe.
#    The 'expand_rows_by_gene' function takes each element in the 'cols_to_list'
#    (which are currently lists) and creates a new row for each index,
#    maintaining the synchronization between Gene IDs, Symbols, and Distances.
df_single_gene = expand_rows_by_gene(df_active_and_diff, list_cols=cols_to_list)

# 2. Display the expanded, gene-centric dataframe
df_single_gene

In [ ]:
# --- High-Confidence Interaction Filtering ---

# Check if the strict link filter is enabled in the global configuration
if top_linked_filter:
    # Filter for high-confidence oligo-gene pairs:
    # 1. Pairs supported by 2 or more evidence sources (eQTL, Hi-C, eRNA, etc.)
    # 2. OR elements located directly within the promoter region ('T' for True)
    df_single_gene = df_single_gene[
        (df_single_gene["num_sources_linking_gene"].astype(int) >= 2) | 
        (df_single_gene["within_promoter"] == "T")
    ]
    
    # Display the refined gene-centric dataframe
    df_single_gene

In [ ]:
# --- LogFC Transformation for Non-Differential Elements ---

# 1. Create a deep copy to ensure the original 'df_single_gene' remains untouched
df_single_gene_zeroed = df_single_gene.copy()

# 2. Neutralize non-differential effects:
#    For any oligo-gene interaction that does not show significant 
#    differential activity, we set its logFC to 0. 
df_single_gene_zeroed.loc[
    df_single_gene_zeroed['differential_activity'] == False,
    'logFC_derived_vs_ancestral'
] = 0

# 3. Display the transformed dataframe
df_single_gene_zeroed

In [ ]:
# --- Statistical Sign Test ---

# 1. Filter the dataset to include ONLY the differentially active elements.
#    Elements with logFC = 0 (set in the previous block) are excluded as they 
#    do not contribute to a sign-based directional bias.
data = df_single_gene_zeroed[df_single_gene_zeroed['differential_activity'] == True]

# 2. Critical Step: Deduplicate by 'id' to ensure each oligo is represented 
#    exactly once in the statistical pool, avoiding bias from one-to-many gene mappings.
data.drop_duplicates(subset=['id'], inplace=True)

# 3. Execute the Sign Test:
#    Analyzes the 'logFC_derived_vs_ancestral' values to check if the balance 
#    between positive and negative changes deviates significantly from 50/50.
run_sign_test(data["logFC_derived_vs_ancestral"].tolist())

In [ ]:
# --- Global LogFC Map Initialization ---

# 1. Initialize/Reset the global fc_map variable.
#    This variable is used as a lookup table by multiple utility functions.
fc_map = None

# 2. Build the mapping dictionary:
#    - Key: Oligo unique identifier ('id')
#    - Value: Allelic effect size ('logFC_derived_vs_ancestral')
#    Note: Non-differential elements already have their values set to 0 in this DataFrame.
fc_map = df_single_gene_zeroed.set_index("id")["logFC_derived_vs_ancestral"].to_dict()

# 3. Verification: Display the first 5 entries of the map to ensure correct formatting
list(fc_map.items())[:5]

In [ ]:
# --- Comprehensive Gene-Level Aggregation ---

df_geneID = (
    # 1. Base Summary: Aggregate all active oligos per gene
    summarize_gene_oligos(df_single_gene_zeroed)
        .rename(
            columns={
                'logFC_list': 'logFC_active',
                'mean_logFC': 'mean_logFC_active',
            }
        )
        # 2. Differential Summary: Merge summaries for only the diff-active subset
        .merge(
            summarize_gene_oligos(
                df_single_gene_zeroed[df_single_gene_zeroed["differential_activity"] == True]
            ).rename(
                columns={
                    'logFC_list': 'logFC_diff',
                    'mean_logFC': 'mean_logFC_diff',
                }
            )[[
                'NCBI_Gene_ID',
                'logFC_diff',
                'mean_logFC_diff',
            ]],
            on='NCBI_Gene_ID',
            how='left', # Use left join to keep all genes even if they have no diff-active enhancers
        )
        # 3. Feature Engineering: Calculate descriptive statistics for both pools
        .assign(
            # Count of enhancers per pool
            num_enhancers_active=lambda d: d['logFC_active'].apply(safe_len),
            num_enhancers_diffActive=lambda d: d['logFC_diff'].apply(safe_len),
            
            # Dispersion of effect sizes (Standard Deviation)
            std_logFC_active=lambda d: d['logFC_active'].apply(safe_std),
            std_logFC_diff=lambda d: d['logFC_diff'].apply(safe_std),
            
            # Directional bias (Percentage of oligos with logFC > 0)
            pct_positive_active=lambda d: d['logFC_active'].apply(categorize_positive_pct),
            pct_positive_diff=lambda d: d['logFC_diff'].apply(categorize_positive_pct),
        )[[
            # 4. Final Column Selection and Ordering
            'NCBI_Gene_ID',
            'num_enhancers_active',
            'logFC_active',
            'mean_logFC_active',
            'std_logFC_active',
            'pct_positive_active',
            'num_enhancers_diffActive',
            'logFC_diff',
            'mean_logFC_diff',
            'std_logFC_diff',
            'pct_positive_diff',
        ]]
)

# Display the final gene-centric summary table
df_geneID

In [ ]:
# --- Quantitative Pool Filtering ---

# Check if a minimum enhancer threshold was specified in the configuration
if min_n > 0:
    # Filter the gene-level dataframe:
    # Retain only genes linked to at least 'min_n' differentially active elements.
    # This ensures that the 'diffActive' pools have sufficient data points for analysis.
    df_geneID = df_geneID[df_geneID['num_enhancers_diffActive'] >= min_n]

# Display the final filtered gene-level dataset
df_geneID

In [ ]:
# --- Genes Trajectory  ---

# 1. Select relevant columns: NCBI_Gene_ID and the list of active logFC values
df_gene_pool = df_geneID.iloc[:, [0, 2]].copy()

# 2. Stochastic Permutation:
#    For each gene, randomly shuffle the order of its active enhancers.
#    This creates a randomized sequence of regulatory effects.
df_gene_pool['logFC_active_scuffled'] = (
    df_gene_pool['logFC_active'].apply(lambda lst: np.random.permutation(lst).tolist())
)

# 3. Cumulative Trajectory Calculation:
#    Compute the cumulative sum (cumsum) of the shuffled effects.
#    This represents a single realization of a 'random walk' through 
#    the gene's regulatory landscape.
df_gene_pool['logFC_active_scuffled_cum'] = (
    df_gene_pool['logFC_active_scuffled'].apply(lambda lst: np.cumsum(lst).tolist())
)

# Display the resulting pools and their simulated trajectories
df_gene_pool


In [ ]:
# --- Background Null Model Preparation ---

# 1. Reference the cleaned, zeroed dataset for background sampling
df_for_random = df_single_gene_zeroed

# 2. Extract and format the oligo-level data:
#    Selects 'id' (oligo_ID) and 'logFC_derived_vs_ancestral'
df_random_oligo_pool = df_for_random.iloc[:, [7, 10]].copy()
df_random_oligo_pool.columns = ['logFC_derived_vs_ancestral', 'oligo_ID']

# 3. Standardize column order for consistency
df_random_oligo_pool = df_random_oligo_pool[['oligo_ID', 'logFC_derived_vs_ancestral']]

# 4. Global Randomization:
#    Shuffle the entire pool of oligos to remove any genomic or experimental ordering.
df_random_oligo_pool = df_random_oligo_pool.sample(frac=1, random_state=None).reset_index(drop=True)

# 5. Deduplication:
#    Ensure each oligo is unique in the background pool (removing redundant gene-oligo links).
df_random_oligo_pool = df_random_oligo_pool.drop_duplicates(subset="oligo_ID")

In [ ]:
# --- Differential Enhancer Abundance and Sign Distribution Analysis ---

# 1. Categorize the sign distribution:
#    Determine the percentage of positive logFC values for diff-active enhancers 
#    and bin them to represent the balance of signs (e.g., all positive, all negative, or mixed).
pct_bins_diff = df_geneID['pct_positive_diff'].apply(bin_pct)

# 2. Visualize Abundance vs. Sign Balance:
#    - vals: Number of diff-active enhancers per gene (Quantitative Abundance).
#    - pct_bins: The categorical distribution of signs within those enhancers.
plot_enhancer_stack_by_pct_grouped_visual(
    vals=df_geneID['num_enhancers_diffActive'],
    pct_bins=pct_bins_diff,
    xlabel="Number of diff-active enhancers per gene",
    title="Distribution of diff-active enhancers per gene\n",
)

In [ ]:
# --- Automated Directory and File Management ---

# 1. Generate a descriptive string based on the current run's configuration.
#    This acts as a 'metadata tag' for the output files.
config_string = f"minN_{min_n}_CondroFilt_{condrocytes_filter}_TopLinkFilt_{top_linked_filter}"

# 2. Create a unique directory for this run's results.
#    The 'exist_ok=True' parameter prevents errors if the folder already exists.
folder_name = f"Run_{config_string}"
os.makedirs(folder_name, exist_ok=True)

# 3. Define structured output paths for both the Gene and Random pools
output_path_genes = os.path.join(folder_name, f"pool_genes_{config_string}.csv")
output_path_random = os.path.join(folder_name, f"pool_random_{config_string}.csv")

# --- Export Processes ---

# Save the Gene-level trajectories 
df_gene_pool.to_csv(output_path_genes, index=False)
print(f"File saved: {output_path_genes}")

# Save the Global Random Background pool 
df_random_oligo_pool.to_csv(output_path_random, index=False)
print(f"File saved: {output_path_random}")